# 02 · Microsoft Agent Framework로 같은 동작 만들기 · 35분

## 학습 목표
- 모델 클라이언트·지침·도구·대화 세션으로 에이전트를 조립합니다.
- 01의 수동 루프를 Framework가 수행하는 것을 관찰합니다.

## 개념
**에이전트 = LLM + 지침 + 도구 + 실행 제어 + 상태**. 도구를 많이 넣는 것 자체가 좋은 설계는 아닙니다. 이번 실습은 읽기 전용 도구만 가진 단일 에이전트입니다.

검증 버전의 `OpenAIChatClient`는 **Responses API**를 사용합니다. 예전 문서의 `OpenAIResponsesClient`는 현재 이름이 아닙니다. `OpenAIChatCompletionClient`는 별도의 Chat Completions 클라이언트입니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


## 실험 · 예측 → 실행 → 관찰 → 변경
**예측:** 01의 `for` 루프가 없어져도 함수가 실행될까요? 후속 질문에서 도시를 아는 이유는 무엇일까요?
**실행:** 기본 `RESET_BEFORE_FOLLOWUP=False`로 두 턴을 실행합니다.
**관찰:** 실제 함수 로그와 첫 답/후속 답을 연결합니다. Framework가 요청→함수→결과 반환을 관리합니다.
**변경:** `RESET_BEFORE_FOLLOWUP=True`만 바꾸고 같은 셀을 다시 실행합니다. 새 세션의 모호한 “방금” 질문에 답할 근거가 있는지 비교하세요.
새 세션에서 우연히 서울을 맞혀도 기억했다는 증거는 아닙니다. 도시는 첫 질문에만 있습니다.


In [ ]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from hanbit_assistant.tools import get_weather as weather_api

prediction = ""  # 새 세션도 방금 도시를 알 수 있을까?
RESET_BEFORE_FOLLOWUP = False  # 변경 실험: True로 바꿔 셀 전체 재실행
executed = []

async def current_weather(city: str) -> str:
    """서울, 도쿄, 런던, 뉴욕의 현재 날씨를 조회한다. 실시간 날씨 질문에 사용한다."""
    executed.append({"tool": "current_weather", "city": city})
    print("실제로 실행된 함수:", "current_weather", city)
    return json.dumps(await weather_api(city), ensure_ascii=False)

async with create_openai_client(settings) as api:
    model_client = OpenAIChatClient(model=settings.model, async_client=api,
        function_invocation_configuration={"max_iterations": 4, "max_function_calls": 3,
                                           "allow_concurrent_invocation": False})
    agent = Agent(client=model_client, name="weather-assistant",
        instructions="한국어 출장 도우미. 현재 날씨는 반드시 도구로 확인하고 기준 시각과 출처를 표시하라. 실패 시 추측하지 마라.",
        tools=[current_weather], default_options={"store": False, "max_tokens": 650})
    session = agent.create_session()
    answer = await agent.run("서울 현재 날씨를 알려줘.", session=session)
    print(answer.text)
    if RESET_BEFORE_FOLLOWUP:
        session = agent.create_session()
    followup = await agent.run("방금 조회한 도시의 이름만 말해줘.", session=session)
    print("후속 답변:", followup.text)

print("실행 기록:", executed)


## 관찰과 체크포인트
- [ ] 내 코드의 직접 루프는 없어도 `current_weather` 로그가 있다.
- [ ] 같은 세션과 새 세션에서 후속 질문의 정보 조건을 구분했다.
- [ ] 도구 수/반복 한도를 제한해도 답변 정확도가 자동 보장되지는 않음을 설명했다.

| 후속 질문 직전 | 답한 도시/확인 질문 | 실제 호출 | 해석 |
|---|---|---|---|
| 세션 유지 | 직접 기록 | 직접 기록 | 직접 기록 |
| 세션 교체 | 직접 기록 | 직접 기록 | 직접 기록 |

**결과 질문:** 세션을 지우는 것이 모델 자체의 학습 지식을 지우는 것과 같은가요?
세션은 이 Python 프로세스의 대화 기록입니다. 서버의 `previous_response_id`로 이어가지 않습니다.

## 연습 · 도구를 하나 더 등록하려면? (빠른 수강생 선택)
아래 풀이를 열기 전에 ① 어떤 함수가 필요한지 ② 인수 타입 ③ `tools` 목록 변경 위치를 적으세요.
기본 실험은 날씨 하나로 완성됩니다. 선택 연습을 하지 않아도 모든 기본 셀을 실행할 수 있습니다.

<details><summary>참고 풀이 · 예시 해설 (선택한 경우에만 복사)</summary>

위 Agent 생성 **전**에 추가합니다.

```python
from hanbit_assistant.tools import convert_currency

async def exchange(amount: float, from_currency: str, to_currency: str) -> str:
    """최근 영업일 참고 환율로 금액을 환산한다."""
    return json.dumps(await convert_currency(amount, from_currency, to_currency), ensure_ascii=False)
```

Agent의 `tools=[current_weather]`를 `tools=[current_weather, exchange]`로 바꾸고,
첫 질문을 `100 USD를 KRW로 환산하고 기준일을 알려줘.`로 바꿉니다.
함수 정의만 추가하고 등록을 빼면 모델은 그 함수를 사용할 수 없습니다.
Framework는 서명과 설명으로 스키마를 만들지만 실제 지원 통화/금액/시간 제한은 함수도 검증해야 합니다.
</details>

### 실패하면
- import 실패: 잠금 파일 그대로 `uv sync --frozen`; 임의 SDK 업그레이드는 하지 않습니다.
- 도구 미실행: 현재 날씨를 명확히 요청했는지, `tools` 등록과 함수 설명을 확인합니다.
- 새 세션인데 답을 맞힘: 우연한 추측과 대화 기록을 구분하세요. 도시를 바꾸어 재실험합니다.
- 제한 초과/연결 오류: 무한 재시도하지 말고 한 번의 짧은 질문으로 줄입니다.

다음은 Search에 사규를 준비하고, 그 검색을 도구로 추가하는 단계입니다.
